In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\geral\Documents\Memoria\experiments\benchmark_final\benchmark_final.csv")

In [11]:
df.columns

Index(['test_mse', 'test_rmse', 'test_mae', 'test_mape', 'test_loss',
       'val_mse', 'val_rmse', 'val_mae', 'val_mape', 'val_loss',
       'train_time_s', 'epochs_run', 'n_params_trainable', 'n_params_total',
       'Dataset_ID', 'Modelo', 'Seed', 'n_train', 'n_val', 'n_test',
       'test_ar_mape'],
      dtype='object')

In [17]:
(df.groupby("Modelo").agg({
    "train_time_s": "mean"
}) * 17 * 3 / 60 / 60 / 24).sum()

train_time_s    50.200585
dtype: float64

In [5]:
import sys
sys.path.append(r"C:\Users\geral\Documents\Memoria\src")

import pandas as pd
import plotly.graph_objects as go
from ts_transformer.inference import ExperimentPredictor

predictor = ExperimentPredictor.from_experiment_dir(r"C:\Users\geral\Documents\Memoria\experiments\test_001")

df = pd.read_csv(r"C:\Users\geral\Documents\Memoria\data\processed\real_data_1.csv")
history = df.iloc[0 : 500 : 3]

In [6]:
preds = predictor.predict(
    past_values=history["valor"],
    past_timestamps=history["timestamp"],
    future_timestamps=[df.iloc[i]["timestamp"] for i in range(500, 600, 3)],
    return_dataframe=True,
)

In [9]:
real_values = df.iloc[500:600:3][["timestamp", "valor"]].reset_index(drop=True)
results = preds.merge(real_values, on="timestamp", how="left", suffixes=("_predicted", "_real"))

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=results["timestamp"], y=results["valor_predicted"], mode="lines+markers", name="Predicted"))
fig.add_trace(go.Scatter(x=results["timestamp"], y=results["valor_real"], mode="lines+markers", name="Real"))
fig.add_trace(go.Scatter(x=history["timestamp"], y=history["valor"], mode="lines+markers", name="History"))
fig.update_layout(title="Predicted vs Real Values", xaxis_title="Timestamp", yaxis_title="Valor")
fig.show()